# オプション課題：画像を水増しすると過学習は減るか（データ拡張）

**問い**：訓練画像を毎回少しずつ変えて（左右反転・回転・明るさ）学習すると，第1回 GW3 試行③の過学習は減るか．

演習3 の 3 つの方法は「モデルや学習のしかた」を変えた．データ拡張は **データの側** を変える 4 つ目の方法で，教科書 第5章ではコードテンプレート②を `data_transforms`（訓練用と検証用）に分けて書く．
教科書には「Cable1_Dataset では回転などを控えめにした方がよい」とある．本当か確かめる．
3〜4 グループで組み，4 つの条件 × 乱数の種を分担する．提出は任意．

1. 実行する前に (1) に予測を書く
2. (2) に自分の担当（`AUG` と `SEED`）を書き，(3) を実行する
3. (4) の手順で CSV を仲間と共有し，(5) で表を作り，(6) の考察を書く（提出）

## (0) グループと番号の設定

In [1]:
# ===== (0) グループと番号の設定 =====
GROUP_ID = 24                 # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_NO = 1                 # ← グループ内の自分の番号 (1〜5)．presenter はこの番号の順に演習ごとに交代する

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

作業フォルダ: /Users/konnotakuto/AI_TD


## (1) 予測（実行する前に書く）

このセルをダブルクリックして書き込む．

- データ拡張なし（none）と，全部入り（all）の検証正解率：none 80％くらい，all 70％くらい。ピンの向きが手がかりなので、回転や明るさまで足すと検証は下がると思う。
- 汎化ギャップ（訓練正解率 − 検証正解率）が一番小さくなる条件：flip。左右反転だけなら形の崩れが小さく、none より訓練データへの寄りが弱いと思う。


In [2]:
# (2) 自分の担当を書く
AUG = "all"     # "none"（なし）/ "flip"（左右反転）/ "rotate"（±10 度回転）/ "all"（反転＋回転＋明るさ）
SEED = 0         # 0〜4 のどれか（同じ AUG を別の SEED で実行すると，値のぶれが見える）

In [3]:
# (3) 学習して，検証データとテストデータの正解率を記録する（40 エポック，30 秒ほど）
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from common.cable import loaders, make_cnn
from common.device import get_device
from common.logger import ResultLogger

AUGS = {
    "none": [],
    "flip": [transforms.RandomHorizontalFlip()],           # 50% の確率で左右反転
    "rotate": [transforms.RandomRotation(10)],             # −10〜+10 度の範囲で回転
    "all": [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.2)],
}
# コードテンプレート②：データ前処理 — 過学習対応（データ拡張は訓練データだけにかける）
data_transforms = {
    "train": transforms.Compose([transforms.Resize((64, 64)), *AUGS[AUG], transforms.ToTensor()]),
    "val": transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()]),
}
device = get_device()
# コードテンプレート③④（GW3 と同じ分け方．SEED では変えない）．各データは CustomDataset に入っているので，変換を差し替えられる
train_loader, val_loader, test_loader, _ = loaders(seed=0)
train_loader.dataset.transform = data_transforms["train"]
plain_train = DataLoader(loaders(seed=0)[0].dataset, batch_size=8)    # 訓練正解率は拡張なしの画像（"val" の変換）で測る
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

model = make_cnn().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def accuracy(loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            correct += (model(images.to(device)).argmax(1).cpu() == labels).sum().item(); total += len(labels)
    return correct / total

for epoch in range(40):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(); criterion(model(images), labels).backward(); optimizer.step()
r = {"train_acc": accuracy(plain_train), "val_acc": accuracy(val_loader), "test_acc": accuracy(test_loader)}
r["gap"] = r["train_acc"] - r["val_acc"]
print(f"{AUG} seed={SEED}  訓練 {r['train_acc'] * 100:.1f}%  検証 {r['val_acc'] * 100:.1f}%  テスト {r['test_acc'] * 100:.1f}%  ギャップ {r['gap'] * 100:.1f}")
logger = ResultLogger(GROUP_ID, f"member{MEMBER_NO}", course="c1", day="d3", exercise="opt3", device=device)
logger.log_many(r, condition=AUG, seed=SEED)
print("書き込み先:", logger.path)

all seed=0  訓練 97.9%  検証 83.3%  テスト 80.0%  ギャップ 14.5
書き込み先: /Users/konnotakuto/AI_TD/results/c1_d3_opt3_24.csv


## (4) 共有する

1. 自分の `results/c1_d3_opt3_<グループ>.csv` を，組んだ仲間全員に送る
2. 受け取った CSV は，すべて `results/opt3_shared/` フォルダに入れる（無ければ作る．同じ名前は末尾に `_2` などを付ける）
3. もう 1 条件実行したいときは，(2) を書き換えて (3) を実行し直す

In [4]:
# (5) 集まった全員分を表にする（何回実行してもよい）．検証・テストは 30 枚ずつなので 1 枚で 3.3 ポイント動く
import glob
import pandas as pd
files = glob.glob(str(ROOT / "results" / "c1_d3_opt3_*.csv")) + glob.glob(str(ROOT / "results" / "opt3_shared" / "*.csv"))
df = pd.concat([pd.read_csv(f) for f in files]).drop_duplicates()
wide = df.pivot_table(index=["condition", "seed", "group_id"], columns="metric_name", values="metric_value").reset_index()
g = wide.groupby("condition")
summary = (g[["train_acc", "val_acc", "test_acc", "gap"]].mean() * 100).add_suffix("_平均")
summary["val_acc_標準偏差"] = g["val_acc"].std() * 100
summary["実行回数"] = g.size()
summary.round(1)

metric_name,train_acc_平均,val_acc_平均,test_acc_平均,gap_平均,val_acc_標準偏差,実行回数
condition,,,,,,
all,99.1,79.3,80.7,19.8,4.3,5
flip,100.0,71.3,86.7,28.7,5.6,5
none,100.0,72.0,82.7,28.0,1.8,5
rotate,100.0,78.0,84.7,22.0,3.8,5


## (6) 考察（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 1〜2 文）．

1. 予測と比べて，当たったこと・外れたこと：予測は、none の検証が80%くらい、all が70%くらいで、all の方が低いと思った。結果は逆で、検証の平均は none が72.0%、all が79.3%だった。汎化ギャップが一番小さいのは flip だと思ったが、flip は28.7ポイントで4条件のうち一番大きく、一番小さいのは all の19.8ポイントだった。
2. データ拡張で汎化ギャップは小さくなったか．どの拡張が一番効いたか：小さくなったのは rotate と all だ。ギャップの平均は none が28.0、flip が28.7、rotate が22.0、all が19.8ポイントで、一番小さくしたのは all だ。all の訓練正解率の平均も、100%から99.1%に少し下がった。
3. 条件の差は，同じ条件を別の SEED で実行したときのぶれより大きいか：検証正解率の標準偏差は none 1.8、flip 5.6、rotate 3.8、all 4.3ポイントで、検証30枚だと1枚で約3.3ポイント動く。none と flip の差0.7ポイントはこのぶれより小さく、none と all の差7.3ポイントは none の標準偏差1.8より大きい。
4. 教科書の「Cable1_Dataset では回転などを控えめにした方がよい」は，この結果で支持されたか．ケーブル画像で拡張が逆効果になりうる理由も書く：この結果だけでは、支持されたとは言えない。±10度の rotate と all は none より検証が高く、ギャップも小さく、大きく回した条件は測っていない。ケーブルはピンの向きが手がかりなので、大きく回すと検証にない姿勢を訓練で作ってずれうるが、今回の±10度ではその害は検証正解率に出ていない。
